In [12]:
import re
import math
import pandas as pd
import matplotlib.pyplot as plt

# --- 1) Robust DMS -> decimal converter ---
_dms_re = re.compile(
    r"""^\s*
        (?P<deg>\d+)\s*°\s*
        (?P<min>\d+)\s*'\s*
        (?P<sec>\d+(?:\.\d+)?)\s*"?\s*
        (?P<hem>[NnSsEeWw])?
        \s*$
    """,
    re.VERBOSE,
)


def dms_to_dd(dms):
    """
    Convert DMS like 39°36'17.12" or 75°15'31.9"W (hemisphere optional) to decimal degrees.
    Returns float or None if it cannot parse.
    """
    if dms is None or (isinstance(dms, float) and math.isnan(dms)):
        return None
    s = str(dms).strip().replace("′", "'").replace("″", '"').replace("’", "'")
    m = _dms_re.match(s)
    if not m:
        return None
    deg = float(m.group("deg"))
    minute = float(m.group("min"))
    sec = float(m.group("sec"))
    hem = m.group("hem")
    dd = deg + minute / 60.0 + sec / 3600.0
    if hem:
        hem = hem.upper()
        if hem in ("S", "W"):
            dd *= -1.0
    return dd


# --- 2) Load your CSV with DMS columns ---
df = pd.read_csv(
    "well_coordinates.csv"
)  # has columns: site_no, latitude_dms, longitude_dms

# Convert to decimal degrees (handle NaNs safely)
df["lat"] = df["latitude_dms"].apply(dms_to_dd)
df["lon"] = df["longitude_dms"].apply(dms_to_dd)

# If hemisphere letter is missing (common on USGS pages), assume US longitudes are West (negative)
# Only apply when hemisphere was missing and lon is positive and within plausible US range
df.loc[df["lon"].notna() & (df["lon"] > 0), "lon"] *= -1

# Drop rows without valid coords
df_plot = df.dropna(subset=["lat", "lon"]).copy()

# --- 3) Try to plot on a US map with cartopy; fall back if not installed ---
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.LambertConformal())

    # Add base features
    ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.5)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"))
    ax.add_feature(cfeature.BORDERS.with_scale("50m"))

    # US bounds
    ax.set_extent([-125, -66.5, 24, 49], crs=ccrs.PlateCarree())

    # Plot points (no labels)
    ax.scatter(df_plot["lon"], df_plot["lat"], s=8, transform=ccrs.PlateCarree())

    plt.title("Well Locations")
    plt.show()

except Exception as e:
    # Fallback: simple lon/lat scatter without a map
    print("cartopy not available or failed; showing plain lon/lat scatter. Reason:", e)
    plt.figure(figsize=(8, 6))
    plt.scatter(df_plot["lon"], df_plot["lat"], s=8)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Well Locations (no basemap)")
    plt.show()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/cartopy/io/__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_1_states_provinces_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)>

<Figure size 1000x600 with 1 Axes>